# Capítulo 5 — Métodos Numéricos em Finanças

Companion em Python inspirado na sequência conceitual do Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander.

> Objetivo: implementar os conceitos matemáticos e financeiros, não reproduzir o texto do livro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import log, exp, sqrt

from quantfinance.linear_algebra import cholesky_factor
from quantfinance.options import black_scholes_call, implied_volatility
from quantfinance.simulation import gbm_terminal_prices

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [ ]:
from scipy import optimize, interpolate, stats

## I.5.2 — Bisseção e Newton–Raphson

In [ ]:
def f(x):
    return x**3 - 2*x - 5

root_bisect = optimize.bisect(f, 2, 3)
root_newton = optimize.newton(f, x0=2.5)

print("Bisseção:", root_bisect)
print("Newton  :", root_newton)

### Volatilidade implícita por Newton

In [ ]:
from scipy.stats import norm

def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_vega(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    return S*norm.pdf(d1)*np.sqrt(T)

def implied_vol_newton(price, S, K, T, r, sigma0=0.2, tol=1e-10, max_iter=100):
    sigma = sigma0
    for _ in range(max_iter):
        diff = bs_call(S,K,T,r,sigma)-price
        if abs(diff) < tol:
            return sigma
        v = bs_vega(S,K,T,r,sigma)
        sigma -= diff/v
    raise RuntimeError("Não convergiu")

target = bs_call(100,100,1,0.05,0.30)
print("Preço alvo manual:", target)
print("Preço via pacote:", black_scholes_call(100,100,1,0.05,0.30))
print("Vol implícita manual:", implied_vol_newton(target,100,100,1,0.05))
print("Vol implícita via pacote:", implied_volatility(target,100,100,1,0.05))

## I.5.3 — Interpolação linear, bilinear e splines

In [ ]:
x = np.array([1, 2, 3, 5, 7, 10], dtype=float)
y = np.array([0.08, 0.085, 0.09, 0.095, 0.097, 0.10])

linear = interpolate.interp1d(x, y)
cubic = interpolate.CubicSpline(x, y)

grid = np.linspace(1, 10, 200)

plt.figure(figsize=(8,4))
plt.plot(x, y, "o", label="Nós")
plt.plot(grid, linear(grid), label="Linear")
plt.plot(grid, cubic(grid), label="Spline cúbica")
plt.legend()
plt.title("Interpolação de uma curva de juros")
plt.show()

## I.5.4 — Otimização e likelihood

In [ ]:
def objective(v):
    x, y = v
    return (x-2)**2 + 2*(y+1)**2

res = optimize.minimize(objective, x0=[0,0])
print(res.x, res.fun)

## I.5.5 — Diferenças finitas e Greeks

In [ ]:
def numerical_delta(S, K, T, r, sigma, h=1e-3):
    return (bs_call(S+h,K,T,r,sigma)-bs_call(S-h,K,T,r,sigma))/(2*h)

def numerical_gamma(S, K, T, r, sigma, h=1e-2):
    return (bs_call(S+h,K,T,r,sigma)-2*bs_call(S,K,T,r,sigma)+bs_call(S-h,K,T,r,sigma))/h**2

print("Delta numérico:", numerical_delta(100,100,1,0.05,0.2))
print("Gamma numérico:", numerical_gamma(100,100,1,0.05,0.2))

## I.5.6 — Árvore binomial para opções europeias e americanas

In [ ]:
def binomial_option(S0, K, T, r, sigma, N=200, option="call", american=False):
    dt = T/N
    u = np.exp(sigma*np.sqrt(dt))
    d = 1/u
    p = (np.exp(r*dt)-d)/(u-d)
    disc = np.exp(-r*dt)

    j = np.arange(N+1)
    ST = S0*(u**j)*(d**(N-j))
    if option == "call":
        V = np.maximum(ST-K, 0.0)
    else:
        V = np.maximum(K-ST, 0.0)

    for i in range(N-1, -1, -1):
        V = disc*(p*V[1:i+2] + (1-p)*V[:i+1])
        if american:
            j = np.arange(i+1)
            S = S0*(u**j)*(d**(i-j))
            intrinsic = np.maximum(S-K,0) if option=="call" else np.maximum(K-S,0)
            V = np.maximum(V, intrinsic)
    return V[0]

print("Call europeia:", binomial_option(100,100,1,0.05,0.2, option="call"))
print("Put americana :", binomial_option(100,100,1,0.05,0.2, option="put", american=True))

## I.5.7 — Monte Carlo

In [ ]:
ST = gbm_terminal_prices(100, 0.08, 0.2, 1, simulations=100_000, seed=123)
print("E[S_T] simulado:", ST.mean())

### Monte Carlo com correlação via Cholesky

In [ ]:
rng = np.random.default_rng(123)

corr = np.array([[1.0, 0.6],
                 [0.6, 1.0]])

C = cholesky_factor(corr)
Z = rng.standard_normal((2, 100_000))
X = C @ Z

print("Correlação simulada:")
print(np.corrcoef(X))

### Simulação multivariada Student-t

In [ ]:
df_t = 5
Z = rng.standard_normal((2, 100_000))
chi = rng.chisquare(df_t, 100_000)
Tdraws = (C @ Z) / np.sqrt(chi/df_t)

print("Correlação amostral Student-t:")
print(np.corrcoef(Tdraws))